# Pandas for Data Engineering — A Practical Guide

This notebook demonstrates essential pandas operations for data engineering workflows, including data loading, type conversion, null handling, outlier filtering, date manipulation, feature creation, merging, and aggregation. It uses an e-commerce orders dataset (~369K rows) and a stores reference table to illustrate real-world data manipulation techniques.

### Preparing the Dataset

Before diving into analysis, we need to load the required libraries and import the raw data. This section covers importing pandas and reading the orders CSV file into a DataFrame.

In [0]:
# Import the pandas library — the primary data manipulation library in Python
import pandas as pd

In [0]:
# Load the orders dataset from the data directory
# The CSV file contains e-commerce order records with 29 columns
orders = pd.read_csv('../data/orders.csv')
orders.head()

,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,order_created_minute,order_created_day,order_created_month,order_created_year,order_moment_created,order_moment_accepted,order_moment_ready,order_moment_collected,order_moment_in_expedition,order_moment_delivering,order_moment_delivered,order_moment_finished,order_metric_collected_time,order_metric_paused_time,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time
0,68405119,3512,5,68405119,68405119,CANCELED,62.70,0.00,NaN,0,1,1,1,2021,1/1/2021 12:01:36 AM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,68405123,3512,5,68405123,68405123,CANCELED,62.70,0.00,NaN,0,4,1,1,2021,1/1/2021 12:04:26 AM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,68405206,3512,5,68405206,68405206,CANCELED,115.50,0.00,NaN,0,13,1,1,2021,1/1/2021 12:13:07 AM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,68405465,3401,5,68405465,68405465,CANCELED,55.90,0.00,NaN,0,19,1,1,2021,1/1/2021 12:19:15 AM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,68406064,3401,5,68406064,68406064,CANCELED,37.90,0.00,NaN,0,26,1,1,2021,1/1/2021 12:26:25 AM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Data Structure Analysis

Once the data is loaded, we inspect its structure — column names, data types, null values, and duplicate rows — to understand what cleaning steps are needed.

In [0]:
# Display rows where the 'order_status' column has null values
# This helps identify missing data that may need to be handled
orders[
    orders['order_status'].isnull()
]

,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,order_created_minute,order_created_day,order_created_month,order_created_year,order_moment_created,order_moment_accepted,order_moment_ready,order_moment_collected,order_moment_in_expedition,order_moment_delivering,order_moment_finished,order_metric_collected_time,order_metric_paused_time,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time


In [0]:
# List all column names in the DataFrame
orders.columns

Index(['order_id', 'store_id', 'channel_id', 'payment_order_id',
       'delivery_order_id', 'order_status', 'order_amount',
       'order_delivery_fee', 'order_delivery_cost', 'order_created_hour',
       'order_created_minute', 'order_created_day', 'order_created_month',
       'order_created_year', 'order_moment_created', 'order_moment_accepted',
       'order_moment_ready', 'order_moment_collected',
       'order_moment_in_expedition', 'order_moment_delivering',
       'order_moment_delivered', 'order_moment_finished',
       'order_metric_collected_time', 'order_metric_paused_time',
       'order_metric_production_time', 'order_metric_walking_time',
       'order_metric_expediton_speed_time', 'order_metric_transit_time',
       'order_metric_cycle_time'],
      dtype='object')

In [0]:
# Drop the 'order_moment_delivered' column
# axis=1 indicates column-level operation (axis=0 would be row-level)
# inplace=True modifies the DataFrame in place without returning a copy
orders.drop('order_moment_delivered', axis=1, inplace=True)

In [0]:
# Count unique values in the 'order_status' column
# Equivalent to SQL: SELECT order_status, COUNT(*) FROM orders GROUP BY order_status
orders['order_status'].value_counts()

order_status
FINISHED    352020
CANCELED     16979
Name: count, dtype: int64

In [0]:
# The 'dtype' property shows the data type of a Series
# The 'astype()' method converts a column to a different data type
print(f"Before: {orders['order_id'].dtype}")
orders['order_id'] = orders['order_id'].astype(int)
print(f"After: {orders['order_id'].dtype}")

In [0]:
# Convert 'order_amount' from string to float for numeric operations
print(f"Before: {orders['order_amount'].dtype}")
orders['order_amount'] = orders['order_amount'].astype(float)
print(f"After: {orders['order_amount'].dtype}")

0          62.7
1          62.7
2         115.5
3          55.9
4          37.9
          ...  
368994     28.0
368995     49.0
368996    128.7
368997    127.4
368998    104.7
Name: order_amount, Length: 368999, dtype: float64

In [0]:
# Remove duplicate rows based on the 'order_id' column
# The 'subset' parameter specifies which columns to use for deduplication
orders.drop_duplicates(subset=['order_id'])

,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,order_created_minute,order_created_day,order_created_month,order_created_year,order_moment_created,order_moment_accepted,order_moment_ready,order_moment_collected,order_moment_in_expedition,order_moment_delivering,order_moment_delivered,order_moment_finished,order_metric_collected_time,order_metric_paused_time,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time
0,68405119,3512,5,68405119,68405119,CANCELADO,62.7,0,7.29,0,1,1,1,2021,1/1/2021 12:01:36 AM,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,68405123,3512,5,68405123,68405123,CANCELADO,62.7,0,7.29,0,4,1,1,2021,1/1/2021 12:04:26 AM,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,68405206,3512,5,68405206,68405206,CANCELADO,115.5,0,7.29,0,13,1,1,2021,1/1/2021 12:13:07 AM,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,68405465,3401,5,68405465,68405465,CANCELADO,55.9,0,7.29,0,19,1,1,2021,1/1/2021 12:19:15 AM,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,68406064,3401,5,68406064,68406064,CANCELADO,37.9,0,7.29,0,26,1,1,2021,1/1/2021 12:26:25 AM,None,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
368994,93139806,2074,5,93139806,93139806,CONCLUÍDO,28,0,0.00,23,59,30,4,2021,4/30/2021 11:59:32 PM,5/1/2021 12:19:50 AM,5/1/2021 12:21:32 AM,5/1/2021 12:22:12 AM,5/1/2021 12:26:19 AM,5/1/2021 12:44:39 AM,None,5/1/2021 1:08:04 AM,0.67,18.33,22,4.8,23.13,23.42,68.53
368995,93139809,707,5,93139809,93139809,CONCLUÍDO,49,11.9,10.04,23,59,30,4,2021,4/30/2021 11:59:39 PM,5/1/2021 12:00:36 AM,5/1/2021 12:26:59 AM,5/1/2021 12:28:24 AM,5/1/2021 12:33:07 AM,5/1/2021 12:36:36 AM,None,5/1/2021 12:57:16 AM,1.42,3.48,27.32,6.13,9.62,20.67,57.6
368996,93139813,631,5,93139813,93139813,CONCLUÍDO,128.7,11.9,9.29,23,59,30,4,2021,4/30/2021 11:59:41 PM,4/30/2021 11:59:59 PM,5/1/2021 12:29:39 AM,5/1/2021 12:41:22 AM,5/1/2021 12:41:24 AM,5/1/2021 12:47:47 AM,None,5/1/2021 1:05:20 AM,11.72,6.38,29.97,11.75,18.13,17.55,65.65
368997,93139814,631,5,93139814,93139814,CONCLUÍDO,127.4,11.9,5.00,23,59,30,4,2021,4/30/2021 11:59:42 PM,5/1/2021 12:00:02 AM,5/1/2021 12:29:57 AM,None,None,5/1/2021 12:31:50 AM,None,5/1/2021 12:45:00 AM,None,None,30.27,None,1.88,13.17,45.32


In [0]:
# Convert 'order_delivery_cost' from string to float
orders['order_delivery_cost'] = orders['order_delivery_cost'].astype(float)

In [0]:
# Fill null values in 'order_delivery_cost' with the column mean (rounded to 2 decimal places)
# The fillna() method replaces NaN values with the specified value
orders['order_delivery_cost'] = orders['order_delivery_cost'].fillna(round(orders['order_delivery_cost'].mean(), 2))

In [0]:
# Replace 'CANCELED' with 'CANCELADO' in the 'order_status' column
# The loc[] accessor allows label-based selection and assignment
orders.loc[
    (orders['order_status'] == 'CANCELED'), 'order_status'
] = 'CANCELADO'

### Anomalies & Outliers

Statistical summary and outlier detection help identify data quality issues. We use `describe()` to get distribution statistics and filter out extreme values that could skew analysis.

In [0]:
# Configure pandas to display float values with 2 decimal places
pd.set_option('display.float_format', '{:.2f}'.format)

In [0]:
# Generate descriptive statistics for the 'order_amount' column
# Shows count, mean, std, min, quartiles, and max
orders['order_amount'].describe()

count    368999.00
mean        105.15
std        2953.11
min           0.00
25%          39.90
50%          71.60
75%         121.90
max     1788306.11
Name: order_amount, dtype: float64

In [0]:
# Filter out outlier orders with amount >= 1000
# These extreme values can distort aggregations and analysis
orders = orders[
    (orders['order_amount'] < 1000)    
]

In [0]:
# Re-check the descriptive statistics after removing outliers
orders['order_amount'].describe()

count   368192.00
mean        95.42
std         85.51
min          0.00
25%         39.90
50%         71.40
75%        121.40
max        999.99
Name: order_amount, dtype: float64

### Date Manipulation

Working with dates is essential for time-based analysis. We convert string timestamps to datetime objects and extract date components for grouping and filtering.

In [0]:
# Convert 'order_moment_created' to datetime, then extract just the date portion
# pd.to_datetime() parses string timestamps into datetime objects
# .dt.date extracts the date component (YYYY-MM-DD), discarding the time
orders['order_moment_created_date'] = pd.to_datetime(orders['order_moment_created']).dt.date

/home/spark-c3d67c9d-6619-4f3a-b888-54/.ipykernel/64/command-5117453766091745-734366287:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders['order_moment_created_date'] = pd.to_datetime(orders['order_moment_created']).dt.date


In [0]:
# Verify that the new date column was added to the DataFrame
orders.columns

Index(['order_id', 'store_id', 'channel_id', 'payment_order_id',
       'delivery_order_id', 'order_status', 'order_amount',
       'order_delivery_fee', 'order_delivery_cost', 'order_created_hour',
       'order_created_minute', 'order_created_day', 'order_created_month',
       'order_created_year', 'order_moment_created', 'order_moment_accepted',
       'order_moment_ready', 'order_moment_collected',
       'order_moment_in_expedition', 'order_moment_delivering',
       'order_moment_delivered', 'order_moment_finished',
       'order_metric_collected_time', 'order_metric_paused_time',
       'order_metric_production_time', 'order_metric_walking_time',
       'order_metric_expediton_speed_time', 'order_metric_transit_time',
       'order_metric_cycle_time', 'order_moment_created_date'],
      dtype='object')

In [0]:
# Display the newly created date column
orders['order_moment_created_date']

0         2021-01-01
1         2021-01-01
2         2021-01-01
3         2021-01-01
4         2021-01-01
             ...    
368994    2021-04-30
368995    2021-04-30
368996    2021-04-30
368997    2021-04-30
368998    2021-04-30
Name: order_moment_created_date, Length: 368192, dtype: object

### Creating New Columns

Feature engineering — creating new columns from existing ones — is a core data preparation step. Here we combine multiple metric columns to derive a lead time metric.

In [0]:
# Display a summary of the DataFrame including column types and non-null counts
orders.info()

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-5117453766091749>, line 1
----> 1 orders.info(nulls=True)

TypeError: DataFrame.info() got an unexpected keyword argument 'nulls'

In [0]:
# Convert time metric columns from string to float for arithmetic operations
orders['order_metric_collected_time'] = orders['order_metric_collected_time'].astype(float)
orders['order_metric_production_time'] = orders['order_metric_production_time'].astype(float)

In [0]:
# Create a new column 'lead_time_product' by summing collection and production times
# This represents the total time from order collection to production completion
orders['lead_time_product'] = orders['order_metric_collected_time'] + orders['order_metric_production_time']
orders['lead_time_product'].describe()


count   316891.00
mean        29.20
std        218.72
min          0.10
25%         11.35
50%         17.20
75%         23.96
max      85667.05
Name: lead_time_product, dtype: float64

In [0]:
# Filter orders where lead time exceeds 1000 (potential anomalies)
orders[
    (orders['lead_time_product']) > 1000
] 

,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,order_created_minute,order_created_day,order_created_month,order_created_year,order_moment_created,order_moment_accepted,order_moment_ready,order_moment_collected,order_moment_in_expedition,order_moment_delivering,order_moment_delivered,order_moment_finished,order_metric_collected_time,order_metric_paused_time,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time,order_moment_created_date,lead_time_product
8,68410055,2181,35,68410055,68410055,CONCLUÍDO,394.80,0.01,6.00,2,32,1,1,2021,1/1/2021 2:32:51 AM,1/1/2021 2:33:00 AM,1/2/2021 6:24:06 PM,1/2/2021 6:30:44 PM,1/2/2021 6:31:15 PM,1/2/2021 6:35:49 PM,None,1/2/2021 6:57:34 PM,6.63,4.55,2391.25,7.17,11.72,21.75,2424.72,2021-01-01,2397.88
1432,68617646,3535,33,68617646,68617646,CONCLUÍDO,259.98,0,3.15,15,4,2,1,2021,1/2/2021 3:04:00 PM,1/4/2021 1:24:50 PM,1/4/2021 4:46:06 PM,1/4/2021 4:46:31 PM,1/4/2021 4:46:32 PM,1/4/2021 5:05:10 PM,None,1/4/2021 6:18:30 PM,0.40,18.62,2982.10,0.43,19.05,73.33,3074.5,2021-01-02,2982.50
2382,68682346,1608,33,68682346,68682346,CONCLUÍDO,259.98,0,3.00,19,18,2,1,2021,1/2/2021 7:18:31 PM,1/4/2021 1:57:47 PM,1/4/2021 9:40:31 PM,1/4/2021 9:41:33 PM,1/4/2021 9:47:07 PM,1/5/2021 1:04:46 PM,None,1/5/2021 1:22:37 PM,1.03,917.67,3022.00,6.6,924.25,17.85,3964.1,2021-01-02,3023.03
5629,68950673,657,20,68950673,68950673,CONCLUÍDO,179.99,0,6.22,21,4,3,1,2021,1/3/2021 9:04:23 PM,1/4/2021 12:26:28 PM,1/4/2021 5:12:08 PM,1/4/2021 5:13:48 PM,1/4/2021 5:18:46 PM,1/4/2021 5:22:36 PM,None,1/4/2021 5:32:40 PM,1.67,3.83,1207.77,6.62,10.47,10.07,1228.3,2021-01-03,1209.44
7075,69106464,1679,35,69106464,69106464,CONCLUÍDO,99.90,9.9,8.00,14,49,4,1,2021,1/4/2021 2:49:36 PM,1/4/2021 2:50:01 PM,1/9/2021 5:13:10 PM,1/9/2021 5:13:23 PM,1/9/2021 5:16:44 PM,1/9/2021 5:20:36 PM,None,1/9/2021 5:30:59 PM,0.22,3.87,7343.57,3.57,7.43,10.37,7361.37,2021-01-04,7343.79
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363903,93104891,3322,35,93104891,93104891,CONCLUÍDO,199.99,9.9,3.45,20,25,29,4,2021,4/29/2021 8:25:28 PM,4/29/2021 8:26:03 PM,4/30/2021 8:38:50 PM,4/30/2021 8:41:17 PM,4/30/2021 8:43:28 PM,4/30/2021 8:50:57 PM,None,4/30/2021 9:39:44 PM,2.45,7.48,1453.37,4.63,12.12,48.78,1514.28,2021-04-29,1455.82
364558,93111738,3322,35,93111738,93111738,CONCLUÍDO,279.99,9.9,12.02,22,39,29,4,2021,4/29/2021 10:39:21 PM,4/29/2021 10:40:00 PM,4/30/2021 8:38:43 PM,4/30/2021 8:40:42 PM,4/30/2021 8:43:26 PM,4/30/2021 9:01:06 PM,None,4/30/2021 9:26:25 PM,1.98,17.67,1319.37,4.72,22.38,25.33,1367.07,2021-04-29,1321.35
365081,93115681,1015,21,93115681,93115681,CONCLUÍDO,44.90,0,5.18,23,57,29,4,2021,4/29/2021 11:57:27 PM,4/30/2021 2:14:33 PM,4/30/2021 2:14:53 PM,4/30/2021 8:02:52 PM,4/30/2021 8:02:53 PM,4/30/2021 9:10:39 PM,None,4/30/2021 9:19:37 PM,347.98,67.77,857.43,348,415.77,8.97,1282.17,2021-04-29,1205.41
366948,93129846,2110,3,93129846,93129846,CONCLUÍDO,190.60,9.9,6.73,18,25,30,4,2021,4/30/2021 6:25:11 PM,4/30/2021 6:25:15 PM,5/1/2021 7:10:43 PM,5/1/2021 7:12:27 PM,None,5/1/2021 7:15:42 PM,None,5/1/2021 8:13:41 PM,1.73,None,1485.53,None,4.98,57.98,1548.5,2021-04-30,1487.26


### Grouping and Selection

Aggregating data by groups is fundamental for reporting and analysis. We use `groupby()` with `agg()` — similar to SQL's `GROUP BY` — to summarize order amounts by date and store.

In [0]:
# Group orders by date and store, then sum the order amounts
# Equivalent to SQL: SELECT order_moment_created_date, store_id, SUM(order_amount)
#                  FROM orders GROUP BY order_moment_created_date, store_id
orders.groupby(['order_moment_created_date', 'store_id']).agg({'order_amount':'sum'}).reset_index()

,order_moment_created_date,store_id,order_amount
0,2021-01-01,1016,731.70
1,2021-01-01,1018,457.48
2,2021-01-01,1037,7683.80
3,2021-01-01,1056,165.40
4,2021-01-01,1057,3803.50
...,...,...,...
33537,2021-04-30,983,569.67
33538,2021-04-30,989,109.90
33539,2021-04-30,99,359.50
33540,2021-04-30,998,158.00


In [0]:
# Load the stores reference dataset
# encoding='latin-1' is used because the file contains non-UTF-8 characters (Portuguese accents)
stores = pd.read_csv('../data/stores.csv', encoding='latin-1')
stores.head()

,store_id,hub_id,store_name,store_segment,store_plan_price,store_latitude,store_longitude
0,3,2,CUMIURI,FOOD,0.00,NaN,NaN
1,6,3,PIMGUCIS DA VIVA,FOOD,0.00,-30.04,-51.20
2,8,3,RASMUR S,FOOD,0.00,-30.04,-51.20
3,53,8,PAPA SUCIS,FOOD,0.00,-22.92,-43.23
4,54,8,VUZPI PAZZIS,FOOD,0.00,-22.92,-43.23


In [0]:
# Extract store_id and store_name into a new DataFrame for joining
stores_name = stores[['store_id', 'store_name']].copy()

In [0]:
# Merge orders with store names using a left join on 'store_id'
# This enriches the orders data with human-readable store names
orders_store = orders.merge(stores_name, how='left', on='store_id')
orders_store.head()

,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,order_created_minute,order_created_day,order_created_month,order_created_year,order_moment_created,order_moment_accepted,order_moment_ready,order_moment_collected,order_moment_in_expedition,order_moment_delivering,order_moment_delivered,order_moment_finished,order_metric_collected_time,order_metric_paused_time,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time,order_moment_created_date,lead_time_product,store_name
0,68405119,3512,5,68405119,68405119,CANCELADO,62.70,0,7.29,0,1,1,1,2021,1/1/2021 12:01:36 AM,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,2021-01-01,NaN,GOI SIPIMUSU
1,68405123,3512,5,68405123,68405123,CANCELADO,62.70,0,7.29,0,4,1,1,2021,1/1/2021 12:04:26 AM,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,2021-01-01,NaN,GOI SIPIMUSU
2,68405206,3512,5,68405206,68405206,CANCELADO,115.50,0,7.29,0,13,1,1,2021,1/1/2021 12:13:07 AM,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,2021-01-01,NaN,GOI SIPIMUSU
3,68405465,3401,5,68405465,68405465,CANCELADO,55.90,0,7.29,0,19,1,1,2021,1/1/2021 12:19:15 AM,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,2021-01-01,NaN,FRUEOU
4,68406064,3401,5,68406064,68406064,CANCELADO,37.90,0,7.29,0,26,1,1,2021,1/1/2021 12:26:25 AM,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,2021-01-01,NaN,FRUEOU


In [0]:
# Group by date and store name, then sum order amounts
# This produces a daily revenue summary per store
order_amount_stores = orders_store.groupby(['order_moment_created_date', 'store_name']).agg({'order_amount':'sum'}).reset_index()

In [0]:
# Export the aggregated result to a CSV file in the data directory
# index=False prevents pandas from writing the row index as a column
order_amount_stores.to_csv('../data/order_amount_stores.csv', index=False)

In [0]:
# Display the final aggregated DataFrame
order_amount_stores.head()

order_moment_created_date,store_name,order_amount
2021-01-01,AR PAZZI,745.4
2021-01-01,AR TRAMUO,74.5
2021-01-01,ARE TRAMUO,125.0
2021-01-01,CILAFIRMAI CIOLA,1090.12
2021-01-01,CIRIROI U CAI,1009.5
2021-01-01,CISI DA PILIP,25.5
2021-01-01,CISI DA URPIOU,52.1
2021-01-01,CISI PIUEUEMI,123.5
2021-01-01,CZALLA PUIMS,399.98
2021-01-01,EAVAMI FIGOI,2109.1
